# 1.&nbsp; Dependencies and Imports

In [ ]:
# Install Dependencies

###
# Unstructured - Parsing and Chunking Documents (pdfs, html, text)
# https://pypi.org/project/unstructured/
# Chroma DB - Internal Vector DB
# Ollama - Local LLM
###

!pip install unstructured "unstructured[all-docs]" chromadb ollama

In [ ]:
# Install and start LLM - Code and Command from Google AI

import os
import subprocess
import time

print("Installing required compression dependencies...")
!apt-get install -y zstd > /dev/null

print("Streaming and extracting Ollama directly to system binaries...")
!curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C /usr

print("Starting Ollama background daemon on your premium GPU...")
ollama_process = subprocess.Popen(
    ["/usr/bin/ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Pulling Llama 3.1 8B (optimized for RAG)...")
subprocess.run(["/usr/bin/ollama", "pull", "llama3.1:8b"])
print("\n System is up and running successfully!")

In [ ]:
# Imports
from google.colab import drive
from pathlib import Path
from unstructured.partition.auto import partition

import chromadb
import ollama
import json

# 2.&nbsp; File Processing
Load and Process Files (Read from disk and Chunk)

In [ ]:
def process_file(filename):
  """Opens and parses files, chunking them into elements.

  Args:
      filename (str): The file with content to be processed.

  Returns:
      tuple[list, list, list]: A tuple containing three lists:
          - **documents**: document chunks in file
          - **metadatas**: document chunk meta data, including page number and filename
          - **ids**: document chunk id

  Coding Assisted via Google AI
  """

  # Opens and Chunks File
  elements = partition(filename=filename)

  documents = []
  metadatas = []
  ids = []



  # Create Section Headers
  current_section_title = None
  for element in elements:
    if element.category == "Title":
        current_section_title = element.text
    if not hasattr(element.metadata, "regex_metadata"):
        element.metadata.regex_metadata = {}

    element.metadata.regex_metadata["section_title"] = current_section_title

  # Creates meta data and document ids
  for index, element in enumerate(elements):
    text_content = str(element)
    if not text_content.strip():
      continue

    documents.append(text_content)
    ids.append(f"doc_chunk_{index}")

    section_title = element.metadata.regex_metadata.get("section_title")
    metadata = {
      "element_type": element.category,
      "section_title": section_title if section_title else "",
      "page_number": element.metadata.page_number if element.metadata.page_number else 1,
      "filename": element.metadata.filename
    }
    metadatas.append(metadata)

  return documents, metadatas, ids

# 3.&nbsp; Vector Database

In [ ]:
class VectorDatabase:
  def __init__(self, path, name):
    self.client = chromadb.PersistentClient(path=path)
    self.collection = self.client.get_or_create_collection(name=name)

  def add_documents(self, documents, metadatas, ids):
    self.collection.add(
      documents=documents,
      metadatas=metadatas,
      ids=ids
    )

  def query_documents(self, query, n_results):
    return self.collection.query(
      query_texts=query,
      n_results=n_results
    )

# 4.&nbsp; Questions System with LLM

In [ ]:
class QuestionsSystem:
  def __init__(self, vector_db, ollama_client):
    self.vector_db = vector_db
    self.ollama_client = ollama_client

  def question_answering(self, user_query):
    """Performs question answering using a vector database and an LLM.

    Args:
      user_query (str): The user's question.

    Returns:
      tuple[str, list]: A tuple containing two objests
          - **answer**: the answer to the question
          - **source**: the source of the answer

    Coding Assisted via Google AI
    """
    db_response = self.vector_db.query_documents(user_query, 4)
    context_str = "\n".join(db_response["documents"][0])

    source = "\n".join([f"{doc_meta['filename']} {doc_meta['section_title']}" for doc_meta in db_response["metadatas"][0]])

    prompt = f"Context:\n{context_str}\n\nQuestion: {user_query}"

    response = self.ollama_client.chat(model='llama3.1:8b', messages=[
        {'role': 'system', 'content': 'Answer using only the given context.'},
        {'role': 'user', 'content': prompt},
    ])
    return response['message']['content'], source

# 5.&nbsp; System Workflow


In [ ]:
# Mount GDrive if using colab

drive.mount('/content/drive')

In [ ]:
def process_questions(questions, folder, vector_db):

  files = [str(f) for f in folder.iterdir() if f.suffix in ('.txt', '.pdf')]
  # Load and Chunk Files
  for file in files:
    documents, metadatas, ids = process_file(file)
    vector_db.add_documents(documents, metadatas, ids)

  # Setup LLM
  q_system = QuestionsSystem(vector_db, ollama.Client())

  # Ask Questions
  received_questions = json.loads(questions)
  results = []
  for q in received_questions:
      result={}
      response = q_system.question_answering(q['question'])
      result['question'] = q['question']
      result['answer'] = response[0]
      result['source'] = response[1]
      results.append(result)

  return results

In [ ]:
# Questions
questions_data = """
[
    {"id": 1, "question": "What is the cost of LEO?", "category": "None"},
    {"id": 2, "question": "What is a small satellite?", "category": "None"},
    {"id": 3, "question": "Why use small satellites?", "category": "None"},
    {"id": 4, "question": "What steps have been taken to reduce launch costs?", "category": "None"},
    {"id": 5, "question": "What is the weight of a small satellite?", "category": "None"}
]
"""

# Create or Load Vector DB
vector_db = VectorDatabase("./chroma_db", "doc_collection")
folder = Path('/content/drive/MyDrive/paper_files/')
response = process_questions(questions_data, folder, vector_db)

print(json.dumps(response, indent=2))